# Context Window Management: Practice Exercise

Build a **token-aware trimming middleware** for an AI agent that manages conversation history based on actual token counts rather than message counts.

**What you'll implement:**
- A `@before_model` middleware function that trims messages when token count exceeds a threshold
- Use `tiktoken` to count tokens accurately
- Keep only the most recent messages that fit within the token budget

**Estimated time:** 10-15 minutes

## Setup

First, let's import the necessary libraries and configure our environment. We'll be using LangChain's `create_agent` function with middleware to manage context.

Key imports:
- `create_agent`: The main function for creating agents with middleware support
- `@before_model`: Decorator for running functions before model invocation
- `InMemorySaver`: Checkpoint storage for conversation persistence
- `tiktoken`: OpenAI's tokenizer library for accurate token counting

In [ ]:
import os
from dotenv import load_dotenv
from typing import Any
import tiktoken

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, RemoveMessage
from langgraph.graph.message import REMOVE_ALL_MESSAGES
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import before_model
from langgraph.runtime import Runtime
from langchain_core.runnables import RunnableConfig

# Load environment variables
load_dotenv()

# Verify API key is loaded
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY not found in environment variables")

print("Setup complete! All required libraries imported successfully.")

## Understanding Middleware in LangChain Agents

Before diving into context management strategies, let's understand **middleware** - a powerful pattern for modifying agent behavior.

### What is Middleware?

Middleware are functions that intercept and potentially modify the agent's execution at specific points. They allow you to:
- Transform state before it reaches the model
- Process outputs after the model responds
- Implement cross-cutting concerns like logging, monitoring, or context management

### Key Middleware Decorators

LangChain provides two main decorators:

1. **`@before_model`**: Runs before the model is invoked
   - Receives current `AgentState` and `Runtime`
   - Can modify messages, add/remove context, etc.
   - Return `dict` with changes or `None` to keep state unchanged

2. **`@after_model`**: Runs after the model responds
   - Receives the model's response
   - Can modify or process the output
   - Useful for logging, validation, post-processing

### The Return Pattern

Middleware functions follow a simple pattern:

```python
@before_model
def my_middleware(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    # Check if modifications are needed
    if no_changes_needed:
        return None  # Keep state as-is
    
    # Make modifications
    modified_state = {...}
    return modified_state  # Apply changes
```

### Passing Middleware to Agents

Middleware is passed as a list to the `middleware` parameter:

```python
agent = create_agent(
    model=model,
    tools=[],
    middleware=[middleware_func1, middleware_func2],  # Applied in order
    checkpointer=InMemorySaver(),
)
```

## Token Counting with tiktoken

In the guided practice, we implemented message-count-based trimming (e.g., keep last 4 messages). However, this approach has a limitation: messages vary greatly in length. A single message could be 10 tokens or 1000 tokens.

**Token-aware trimming** is more precise because it:
- Accounts for actual token usage, not just message count
- Prevents context overflow more reliably
- Allows you to maximize context usage within limits

### Using tiktoken

`tiktoken` is OpenAI's tokenizer library. Here's how to use it:

In [ ]:
# Get the tokenizer for GPT-4 models
encoding = tiktoken.encoding_for_model("gpt-4o")

# Example: Count tokens in a string
text = "Hello, how can I help you today?"
tokens = encoding.encode(text)
print(f"Text: '{text}'")
print(f"Token count: {len(tokens)}")
print(f"Tokens: {tokens}")

# Longer text example
long_text = "The quick brown fox jumps over the lazy dog. This is a classic pangram used to test typewriters and fonts."
long_tokens = encoding.encode(long_text)
print(f"\nLonger text token count: {len(long_tokens)}")

## Your Task: Implement Token-Aware Trimming

Implement a middleware function that trims messages based on **token count** rather than message count.

### Requirements

1. Count the total tokens across all messages in the conversation
2. If total tokens exceed `MAX_TOKENS`, trim older messages
3. Keep the most recent messages that fit within the token budget
4. Use `RemoveMessage(id=REMOVE_ALL_MESSAGES)` followed by the messages to keep

### Helper Function Provided

A `count_message_tokens()` function is provided below to help you count tokens in a message.

In [ ]:
# Configuration
MAX_TOKENS = 500  # Trigger trimming when total tokens exceed this

# Initialize tokenizer
encoding = tiktoken.encoding_for_model("gpt-4o")


def count_message_tokens(message) -> int:
    """
    Count the number of tokens in a message.
    
    Args:
        message: A LangChain message object (HumanMessage, AIMessage, etc.)
    
    Returns:
        int: Number of tokens in the message content
    """
    return len(encoding.encode(message.content))


# Test the helper function
test_msg = HumanMessage(content="Hi, my name is Sajal and I'm planning a trip to Japan.")
print(f"Test message tokens: {count_message_tokens(test_msg)}")

## Implement the Token-Aware Trimming Middleware

Complete the middleware function below.

**Algorithm:**
1. Calculate total tokens across all messages
2. If total <= `MAX_TOKENS`, return `None` (no trimming needed)
3. If total > `MAX_TOKENS`:
   - Iterate through messages from most recent to oldest
   - Add messages to a "keep" list while their cumulative tokens stay under `MAX_TOKENS`
   - Return the trimmed message list

**Output format:**
```python
{
    "messages": [
        RemoveMessage(id=REMOVE_ALL_MESSAGES),
        *messages_to_keep  # In chronological order (oldest first)
    ]
}
```

In [ ]:
@before_model
def token_aware_trim(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """
    Trim conversation history based on token count.
    
    This middleware:
    - Counts total tokens across all messages
    - If over MAX_TOKENS, keeps only the most recent messages that fit
    - Preserves messages in chronological order
    
    Args:
        state: Current agent state containing messages list
        runtime: Runtime object for agent execution context
    
    Returns:
        dict with modified messages if trimming needed, None otherwise
    """
    messages = state["messages"]
    
    # TODO: Step 1 - Calculate total tokens across all messages
    # Hint: Use count_message_tokens() for each message and sum them
    total_tokens = None  # Replace with your implementation
    
    # TODO: Step 2 - Check if trimming is needed
    # If total_tokens <= MAX_TOKENS, return None
    pass  # Replace with your implementation
    
    # TODO: Step 3 - Build list of messages to keep (most recent first)
    # Iterate through messages in reverse order (most recent first)
    # Add each message while cumulative tokens stay under MAX_TOKENS
    messages_to_keep = []  # Replace with your implementation
    
    # TODO: Step 4 - Reverse to restore chronological order and return
    # Return dict with RemoveMessage + messages_to_keep in correct order
    pass  # Replace with your implementation

## Create the Agent with Your Middleware

Create an agent using your token-aware trimming middleware.

In [ ]:
# Create model instance
model = ChatOpenAI(model="gpt-4o")

# Create agent with token-aware trimming middleware
agent = create_agent(
    model=model,
    tools=[],  # No tools for this demonstration
    middleware=[token_aware_trim],
    checkpointer=InMemorySaver(),
)

print("Agent with token-aware trimming middleware created!")
print(f"Will trim when total tokens exceed: {MAX_TOKENS}")

## Test Your Implementation

Let's simulate a conversation and verify that token-aware trimming works correctly.

In [ ]:
# Configuration with thread ID for conversation persistence
config: RunnableConfig = {"configurable": {"thread_id": "token_trim_demo"}}

# Simulate a multi-turn conversation
conversation = [
    "Hi! My name is Sajal and I'm planning a trip to Japan.",
    "I'm interested in visiting Tokyo first.",
    "What are the must-see places in Tokyo?",
    "How about food recommendations?",
    "What's the best way to get around Tokyo?",
    "Should I get a JR Pass?",
]

print("Starting conversation with token-aware trimming agent...")
print("=" * 80)

for i, msg in enumerate(conversation, 1):
    result = agent.invoke(
        {"messages": [HumanMessage(content=msg)]}, 
        config
    )
    
    # Get the last AI message
    ai_response = result['messages'][-1].content
    
    print(f"\nTurn {i}:")
    print(f"User: {msg}")
    print(f"Agent: {ai_response[:100]}...")
    print("-" * 80)

print("\nConversation complete!")

## Verify Trimming Behavior

Check what messages remain in memory and their token counts.

In [ ]:
# Get the current state to inspect memory
state = agent.get_state(config)
messages = state.values["messages"]

print(f"Total messages in memory: {len(messages)}")

total_tokens = 0
print(f"\nMessage breakdown:")
for i, msg in enumerate(messages, 1):
    msg_type = msg.__class__.__name__
    tokens = count_message_tokens(msg)
    total_tokens += tokens
    content_preview = msg.content[:50] if len(msg.content) > 50 else msg.content
    print(f"  {i}. {msg_type} ({tokens} tokens): {content_preview}...")

print(f"\nTotal tokens in memory: {total_tokens}")
print(f"Token budget (MAX_TOKENS): {MAX_TOKENS}")

if total_tokens <= MAX_TOKENS:
    print("\nToken-aware trimming is working correctly!")
else:
    print("\nWarning: Total tokens exceed the budget. Check your implementation.")

## Test Information Loss

Since we're trimming based on tokens (keeping recent messages), early conversation context will be lost.

In [ ]:
# Test if the agent remembers the name from the first message
result = agent.invoke(
    {"messages": [HumanMessage(content="What's my name?")]},
    config
)

print("Testing recall of early conversation...")
print(f"\nUser: What's my name?")
print(f"\nAgent: {result['messages'][-1].content}")

print("\n" + "=" * 80)
print("EXPECTED BEHAVIOR:")
print("The agent likely won't remember the name because the first message")
print("was trimmed to stay within the token budget.")
print("\nThis demonstrates the trade-off of trimming: fast and cheap,")
print("but information from older messages is permanently lost.")
print("=" * 80)